# Données

## Importation des packages

In [1]:
import numpy as np
import matplotlib.pyplot as plt
import pandas as pd
pd.set_option('display.max_rows', 500)

import requests
from bs4 import BeautifulSoup
import os
import s3fs
import ast

## Lecture des fichiers movies_metadata.csv et credits.csv

Les données de movies_metadata.csv et credits.csv sont des données trouvées sur Kaggle qui centralisent des informations diverses sur des films sortis avant juillet 2017. 

Les variables de movies_metadata.csv sont :
adult : signification de la varible non connue
belongs to collection : si le film appartient à une série de film, la variable renseigne les films faisant partie de cette série
budget : budget du film
genres : genres du film
homepage : lien vers le site officiel du film s'il y en a un
id et imdb_id : identifiants du film
original_language : langue originale du film
original_title : titre original du film
overview : résumé du film
popularity : popularité du film sur IMDB
poster_path : lien vers l'affiche du film
production_countries : pays de production du film
production_companies : compagnies de production du film
release_date : date de sortie du film
revenue : recettes du film
runtime : durée du film
spoken_languages : langues parlées dans le film en version originale
status : si le film est sorti, prévu, annulé, en production etc
tagline : catchphrase du film
title : titre anglophone du film
video : False si le film est sorti au cinéma, True s'il est sorti directement sur Internet et qu'il n'a pas été diffusé au cinéma

Les variables de credits.csv sont :
cast : casting du film sous forme de liste de dictionnaires
crew : équipe du film sous forme de liste de dictionnaires

Nous souhaitons à partir de ces données prédire la note de nouveaux films, voir quelles sont les variables les plus décisives pour prédire si un film sera ien reçu par le public et ainsi remarquer (ou non) la prévisibilité du succès d'un film.

Nous pourrons pondérer l'erreur de prévision avec la variable vote_count et faire de la classification non supervisée dans les stats descriptives

In [2]:
os.environ['AWS_S3_ENDPOINT']
S3_ENDPOINT_URL = 'http://' + os.environ['AWS_S3_ENDPOINT']
fs = s3fs.S3FileSystem(client_kwargs = {'endpoint_url' :S3_ENDPOINT_URL })
fs.ls('mlepennec-ensae')
BUCKET = 'mlepennec-ensae'
FILE_KEY_S3 = '/movies_metadata.csv'
FILE_PATH_S3 = BUCKET+FILE_KEY_S3
with fs.open(FILE_PATH_S3, mode = "rb") as file_in : 
    data_movies = pd.read_csv(file_in,sep=',', header=0)

/tmp/ipykernel_14698/2309787388.py:9: DtypeWarning: Columns (10) have mixed types. Specify dtype option on import or set low_memory=False.
  data_movies = pd.read_csv(file_in,sep=',', header=0)


In [3]:
FILE_KEY_S3 = '/credits.csv'
FILE_PATH_S3 = BUCKET+FILE_KEY_S3
with fs.open(FILE_PATH_S3, mode = "rb") as file_in : 
    data_credits = pd.read_csv(file_in,sep=',', header=0)

Après première exploration des données, nous avons décidé d'enlever les variables suivantes : adult, homepage, overview, popularity, poster_path, spoken_languages et tagline. En effet, la variable adult présente presque toujours la modalité False et semble avoir peu d'intérêt. La variable homepage renseigne le lien vers le site officiel du film s'il y en a un. La variable overview contient les résumés des films du dataframe, ce qui peut être intéressant à exploiter mais nous avons décidé de ne pas le faire. La variable popularity est la popularité du film sur IMDB au moment où les données ont été extraites, c'est donc une variable qui n'est pas statique et qui est calculée directement par IMDB d'une façon que nous ignorons donc nous ne souhaitons pas la prendre en compte. La variable poster_path indique le lien vers l'affiche du film, nous n'en avons pas besoin. La variable spoken_languages indique les langues parlées durant le film en version originale, nous considérons que cette variable est redondante par rapport à la variable original_language. Enfin la variable tagline indique la catchphrase du film, ce qui est à nos yeux peu utile également.

Nous allons retraiter certaines variables. Par exemple, la variable belongs_to_collection sera transformée en booléen (1 si le film correspond à une série de films, 0 sinon) à laquelle nous ajouterons une variable avec le nombre de films précédents de la série ainsi que la note du film précédent. La variable genres sera décomposée en plusieurs variables genre_1, genre_2 etc. Ce genre de décomposition sera également nécessaire pour les variables production_countries et production_companies

Les variables budget et runtime présentent des valeurs manquantes, que nous allons essayer de compléter avec du web scraping.

Nous allons nous concentrer sur les films qui sont déjà sortis en salle (status = Released et video=False)

Les données du fichier credits.csv vont nous permettre d'ajouter les acteurs principaux et le réalisateur du film à notre jeu de données. Nous souhaitons ajouter des variables relatives à la popularité des acteurs et du réalisateur via du web scraping.


In [4]:
data_movies.original_language.value_counts()

original_language
en       32269
fr        2438
it        1529
ja        1350
de        1080
es         994
ru         826
hi         508
ko         444
zh         409
sv         384
pt         316
cn         313
fi         297
nl         248
da         225
pl         219
tr         150
cs         130
el         113
no         106
fa         101
hu         100
ta          78
th          76
he          67
sr          63
ro          57
te          45
ar          39
ml          36
xx          33
bn          29
hr          29
mr          25
is          24
et          24
tl          23
id          20
ka          18
lv          18
sl          17
uk          16
bs          14
ca          12
ab          10
bg          10
vi          10
lt           9
sk           9
ur           8
nb           6
mk           5
wo           5
ms           5
sh           5
sq           5
ky           3
ku           3
bm           3
eu           3
kk           3
kn           3
bo           2
pa           2
af     

In [5]:
data_movies_df = data_movies[data_movies['video'] == False]
data_movies_df = data_movies_df[data_movies_df['status'] == 'Released']
data_movies_df = data_movies_df.drop(columns=['adult', 'homepage', 'overview', 'popularity', 'poster_path', 'tagline', 'status', 'video'])
data_movies_df = data_movies_df.dropna(subset= ['release_date'])
data_movies_df = data_movies_df.dropna(subset= ['imdb_id'])
data_movies_df = data_movies_df.dropna(subset= ['original_language'])

In [6]:
data_movies_df.info()

<class 'pandas.core.frame.DataFrame'>
Index: 44825 entries, 0 to 45465
Data columns (total 16 columns):
 #   Column                 Non-Null Count  Dtype  
---  ------                 --------------  -----  
 0   belongs_to_collection  4460 non-null   object 
 1   budget                 44825 non-null  object 
 2   genres                 44825 non-null  object 
 3   id                     44825 non-null  object 
 4   imdb_id                44825 non-null  object 
 5   original_language      44825 non-null  object 
 6   original_title         44825 non-null  object 
 7   production_companies   44825 non-null  object 
 8   production_countries   44825 non-null  object 
 9   release_date           44825 non-null  object 
 10  revenue                44825 non-null  float64
 11  runtime                44588 non-null  float64
 12  spoken_languages       44825 non-null  object 
 13  title                  44825 non-null  object 
 14  vote_average           44825 non-null  float64
 15  vote_co

In [7]:
missing_percentage = data_movies_df.isna().sum()

print('MISSING VALUES :')
if missing_percentage[missing_percentage != 0].empty:
    print('No')
else:
    print(missing_percentage[missing_percentage != 0].sort_values(ascending=False))

MISSING VALUES :
belongs_to_collection    40365
runtime                    237
dtype: int64


On ajoute au dataframe les différents url wikipédia possibles pour un film (selon le nom du film, il faut parfois ajouter film ou film + année de sortie à l'url wikipédia pour tomber sur la bonne page wiki)

In [8]:
url_wikipedia_fr = "https://fr.wikipedia.org/wiki/"
url_wikipedia_en = "https://en.wikipedia.org/wiki/"
data_movies_df['url'] = url_wikipedia_en + data_movies_df.title.str.replace(" ", "_")
data_movies_df['url_film'] = url_wikipedia_en + data_movies_df.title.str.replace(" ", "_") + "_(film)"
data_movies_df['release_year'] = data_movies_df.release_date.str[:4]
data_movies_df['url_film_date'] = url_wikipedia_en + data_movies_df.title.str.replace(" ", "_") + "_(film,_" + data_movies_df.release_year + ")"
data_movies_df['id'] = pd.to_numeric(data_movies_df['id'])


On joint les dataframes movies et credits pour ajouter le casting et l'équipe du film.

In [9]:
data_movies_credits = data_movies_df.merge(data_credits, left_on='id', right_on='id')
data_movies_credits


,belongs_to_collection,budget,genres,id,imdb_id,original_language,original_title,production_companies,production_countries,release_date,...,spoken_languages,title,vote_average,vote_count,url,url_film,release_year,url_film_date,cast,crew
0,"{'id': 10194, 'name': 'Toy Story Collection', ...",30000000,"[{'id': 16, 'name': 'Animation'}, {'id': 35, '...",862,tt0114709,en,Toy Story,"[{'name': 'Pixar Animation Studios', 'id': 3}]","[{'iso_3166_1': 'US', 'name': 'United States o...",1995-10-30,...,"[{'iso_639_1': 'en', 'name': 'English'}]",Toy Story,7.7,5415.0,https://en.wikipedia.org/wiki/Toy_Story,https://en.wikipedia.org/wiki/Toy_Story_(film),1995,"https://en.wikipedia.org/wiki/Toy_Story_(film,...","[{'cast_id': 14, 'character': 'Woody (voice)',...","[{'credit_id': '52fe4284c3a36847f8024f49', 'de..."
1,NaN,65000000,"[{'id': 12, 'name': 'Adventure'}, {'id': 14, '...",8844,tt0113497,en,Jumanji,"[{'name': 'TriStar Pictures', 'id': 559}, {'na...","[{'iso_3166_1': 'US', 'name': 'United States o...",1995-12-15,...,"[{'iso_639_1': 'en', 'name': 'English'}, {'iso...",Jumanji,6.9,2413.0,https://en.wikipedia.org/wiki/Jumanji,https://en.wikipedia.org/wiki/Jumanji_(film),1995,"https://en.wikipedia.org/wiki/Jumanji_(film,_1...","[{'cast_id': 1, 'character': 'Alan Parrish', '...","[{'credit_id': '52fe44bfc3a36847f80a7cd1', 'de..."
2,"{'id': 119050, 'name': 'Grumpy Old Men Collect...",0,"[{'id': 10749, 'name': 'Romance'}, {'id': 35, ...",15602,tt0113228,en,Grumpier Old Men,"[{'name': 'Warner Bros.', 'id': 6194}, {'name'...","[{'iso_3166_1': 'US', 'name': 'United States o...",1995-12-22,...,"[{'iso_639_1': 'en', 'name': 'English'}]",Grumpier Old Men,6.5,92.0,https://en.wikipedia.org/wiki/Grumpier_Old_Men,https://en.wikipedia.org/wiki/Grumpier_Old_Men...,1995,https://en.wikipedia.org/wiki/Grumpier_Old_Men...,"[{'cast_id': 2, 'character': 'Max Goldman', 'c...","[{'credit_id': '52fe466a9251416c75077a89', 'de..."
3,NaN,16000000,"[{'id': 35, 'name': 'Comedy'}, {'id': 18, 'nam...",31357,tt0114885,en,Waiting to Exhale,[{'name': 'Twentieth Century Fox Film Corporat...,"[{'iso_3166_1': 'US', 'name': 'United States o...",1995-12-22,...,"[{'iso_639_1': 'en', 'name': 'English'}]",Waiting to Exhale,6.1,34.0,https://en.wikipedia.org/wiki/Waiting_to_Exhale,https://en.wikipedia.org/wiki/Waiting_to_Exhal...,1995,https://en.wikipedia.org/wiki/Waiting_to_Exhal...,"[{'cast_id': 1, 'character': ""Savannah 'Vannah...","[{'credit_id': '52fe44779251416c91011acb', 'de..."
4,"{'id': 96871, 'name': 'Father of the Bride Col...",0,"[{'id': 35, 'name': 'Comedy'}]",11862,tt0113041,en,Father of the Bride Part II,"[{'name': 'Sandollar Productions', 'id': 5842}...","[{'iso_3166_1': 'US', 'name': 'United States o...",1995-02-10,...,"[{'iso_639_1': 'en', 'name': 'English'}]",Father of the Bride Part II,5.7,173.0,https://en.wikipedia.org/wiki/Father_of_the_Br...,https://en.wikipedia.org/wiki/Father_of_the_Br...,1995,https://en.wikipedia.org/wiki/Father_of_the_Br...,"[{'cast_id': 1, 'character': 'George Banks', '...","[{'credit_id': '52fe44959251416c75039ed7', 'de..."
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
44893,NaN,0,"[{'id': 18, 'name': 'Drama'}, {'id': 28, 'name...",30840,tt0102797,en,Robin Hood,"[{'name': 'Westdeutscher Rundfunk (WDR)', 'id'...","[{'iso_3166_1': 'CA', 'name': 'Canada'}, {'iso...",1991-05-13,...,"[{'iso_639_1': 'en', 'name': 'English'}]",Robin Hood,5.7,26.0,https://en.wikipedia.org/wiki/Robin_Hood,https://en.wikipedia.org/wiki/Robin_Hood_(film),1991,https://en.wikipedia.org/wiki/Robin_Hood_(film...,"[{'cast_id': 1, 'character': 'Sir Robert Hode'...","[{'credit_id': '52fe44439251416c9100a899', 'de..."
44894,NaN,0,"[{'id': 18, 'name': 'Drama'}]",111109,tt2028550,tl,Siglo ng Pagluluwal,"[{'name': 'Sine Olivia', 'id': 19653}]","[{'iso_3166_1': 'PH', 'name': 'Philippines'}]",2011-11-17,...,"[{'iso_639_1': 'tl', 'name': ''}]",Century of Birthing,9.0,3.0,https://en.wikipedia.org/wiki/Century_of_Birthing,https://en.wikipedia

On retraite les colonnes cast et crew pour que Python les reconnaissent en tant que liste de dictionnaires.

In [10]:
data_movies_credits['cast'] = data_movies_credits['cast'].apply(ast.literal_eval)
data_movies_credits['crew'] = data_movies_credits['crew'].apply(ast.literal_eval)

On ajoute les colonnes correspondant aux 4 acteurs principaux du film et une colonne pour le réalisateur du film.

In [11]:
data_movies_credits['acteur_1'] = data_movies_credits['cast'].apply(
    lambda lst: lst[0]['name'] if isinstance(lst, list) and len(lst) > 0 else None
)
data_movies_credits['acteur_2'] = data_movies_credits['cast'].apply(
    lambda lst: lst[1]['name'] if isinstance(lst, list) and len(lst) > 1 else None
)
data_movies_credits['acteur_3'] = data_movies_credits['cast'].apply(
    lambda lst: lst[2]['name'] if isinstance(lst, list) and len(lst) > 2 else None
)
data_movies_credits['acteur_4'] = data_movies_credits['cast'].apply(
    lambda lst: lst[3]['name'] if isinstance(lst, list) and len(lst) > 3 else None
)

data_movies_credits['realisateur'] = data_movies_credits['crew'].apply(
    lambda lst: lst['job' == 'Director']['name'] if isinstance(lst, list) and len(lst) > 0 else None
)


On enlève les lignes où il n'y a pas d'acteurs.

In [ ]:
#data_movies_credits = data_movies_credits[data_movies_credits['cast'].apply(lambda x: len(x) != 0)]
#data_movies_credits

In [24]:
data=data_movies_credits[data_movies_credits["budget"]=="0"]
data[['title', 'url', 'url_film', 'url_film_date']]

,title,url,url_film,url_film_date
2,Grumpier Old Men,https://en.wikipedia.org/wiki/Grumpier_Old_Men,https://en.wikipedia.org/wiki/Grumpier_Old_Men...,https://en.wikipedia.org/wiki/Grumpier_Old_Men...
4,Father of the Bride Part II,https://en.wikipedia.org/wiki/Father_of_the_Br...,https://en.wikipedia.org/wiki/Father_of_the_Br...,https://en.wikipedia.org/wiki/Father_of_the_Br...
7,Tom and Huck,https://en.wikipedia.org/wiki/Tom_and_Huck,https://en.wikipedia.org/wiki/Tom_and_Huck_(film),https://en.wikipedia.org/wiki/Tom_and_Huck_(fi...
11,Dracula: Dead and Loving It,https://en.wikipedia.org/wiki/Dracula:_Dead_an...,https://en.wikipedia.org/wiki/Dracula:_Dead_an...,https://en.wikipedia.org/wiki/Dracula:_Dead_an...
12,Balto,https://en.wikipedia.org/wiki/Balto,https://en.wikipedia.org/wiki/Balto_(film),"https://en.wikipedia.org/wiki/Balto_(film,_1995)"
...,...,...,...,...
44893,Robin Hood,https://en.wikipedia.org/wiki/Robin_Hood,https://en.wikipedia.org/wiki/Robin_Hood_(film),https://en.wikipedia.org/wiki/Robin_Hood_(film...
44894,Century of Birthing,https://en.wikipedia.org/wiki/Century_of_Birthing,https://en.wikipedia.org/wiki/Century_of_Birth...,https://en.wikipedia.org/wiki/Century_of_Birth...
44895,Betrayal,https://en.wikipedia.org/wiki/Betrayal,https://en.wikipedia.org/wiki/Betrayal_(film),"https://en.wikipedia.org/wiki/Betrayal_(film,_..."
44896,Satan Triumphant,https://en.wikipedia.org/wiki/Satan_Triumphant,https://en.wikipedia.org/wiki/Satan_Triumphant...,https://en.wikipedia.org/wiki/Satan_Triumphant...


In [25]:
headers = {
    'User-Agent': 'Mozilla/5.0 (Windows NT 10.0; Win64; x64) AppleWebKit/537.36 (KHTML, like Gecko) Chrome/117.0.0.0 Safari/537.36'
}

Fonction pour trouver le budget d'un film avec l'url wikipédia

In [27]:
def extraire_budget_depuis_wikipedia(url):
    try:
        # Charger la page
        response = requests.get(url, headers=headers)
        response.raise_for_status()

        # Parser le HTML
        soup = BeautifulSoup(response.content, 'html.parser')

        # Trouver l'infobox (il peut y avoir plusieurs classes, mais 'infobox' est souvent commun)
        infobox = soup.find('table', class_='infobox')

        if infobox is None:
            return None  # Pas d'infobox trouvée

        # Chercher les lignes de l'infobox
        rows = infobox.find_all('tr')

        for row in rows:
            header = row.find('th')
            if header and 'budget' in header.get_text(strip=True).lower():
                # Trouver la cellule contenant la valeur
                value_cell = row.find('td')
                if value_cell:
                    return value_cell.get_text(separator=" ", strip=True)

        return None  # Pas de ligne contenant "budget"

    except Exception as e:
        print(f"Erreur lors du traitement de {url}: {e}")
        return None


In [ ]:
data['budget_url'] = data['url'].apply(extraire_budget_depuis_wikipedia)
data['budget_url_film'] = data['url_film'].apply(extraire_budget_depuis_wikipedia)
data['budget_url_film_date'] = data['url_film_date'].apply(extraire_budget_depuis_wikipedia)

# Ton DataFrame à sauvegarder
# Exemple : df = pd.DataFrame({'col1': [1, 2], 'col2': ['a', 'b']})
BUCKET = 'mlepennec-ensae'

FILE_OUT_S3 = '/movies_budget.csv'  # Chemin dans le bucket
FILE_OUT_PATH = BUCKET + FILE_OUT_S3       # Chemin complet S3

# Écriture vers S3
with fs.open(FILE_OUT_PATH, mode='w') as f_out:
    data.to_csv(f_out, index=False)